# 05 · Interpolação: Newton, baricêntrica e splines

**Objetivo pedagógico:** Reproduzir polinômios, escolher nós e separar erro de aproximação de erro aritmético.

**Luigi Scofano de Araujo · USP 13680334**

Reelaboração de estudo de um tema presente no acervo recuperado de 2025. Esta edição foi preparada em 2026 com assistência de IA. Não é uma cópia autenticada de uma entrega nem uma afirmação de conclusão das questões originais. Os arquivos recuperados estão em `acervo/`, com créditos e limitações em `docs/PROCEDENCIA.md`.

Execute as células em ordem, em kernel novo. Os dados são locais/sintéticos e as conclusões numéricas são verificadas por asserções; gráficos sozinhos não certificam convergência.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd()/"numericos").is_dir() else Path.cwd().parent
if not (ROOT/"numericos").is_dir():
    raise RuntimeError("Execute a partir da raiz do repositório ou de notebooks/.")
sys.path.insert(0, str(ROOT))
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=7, suppress=True)
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True})
FIGURAS = ROOT/"assets"/"figuras"
FIGURAS.mkdir(parents=True, exist_ok=True)
def mostrar(nome):
    plt.tight_layout()
    plt.savefig(FIGURAS/(nome+".svg"), bbox_inches="tight", metadata={"Date": None})
    plt.show()
    plt.close("all")

## 1. Diferenças divididas e avaliação aninhada
A forma de Newton usa coeficientes construídos de diferenças divididas. O nome “Newton” aqui se refere à interpolação, não ao método iterativo de encontrar raízes. Um polinômio cúbico é reproduzido por quatro nós distintos, salvo arredondamento.

In [2]:
from numericos.interpolacao import diferencas_divididas,avaliar_newton,baricentrica,nos_chebyshev
nodes=np.array([-1.,0.,1.,2.]);f=lambda x:x**3-2*x+1
values=f(nodes);coefficients=diferencas_divididas(nodes,values)
grid=np.linspace(-1,2,301)
pn=avaliar_newton(grid,nodes,coefficients)
pb=baricentrica(grid,nodes,values)
assert np.allclose(pn,f(grid),atol=1e-12)
assert np.allclose(pb,pn,atol=1e-12)
assert np.array_equal(baricentrica(nodes,nodes,values),values)
print("coeficientes:",coefficients,"erro máximo:",np.max(abs(pb-f(grid))))

coeficientes: [ 2. -1.  0.  1.] erro máximo: 8.881784197001252e-16


## 2. Fenômeno de Runge e nós de Chebyshev
Compare f(x)=1/(1+25x²) em [−1,1] usando 17 nós equidistantes ou de Chebyshev-Lobatto. As extremidades tendem a ser críticas para os nós uniformes. Esta comparação é de uma função e de um conjunto finito de graus; não transforma Chebyshev em solução universal para qualquer dado ruidoso.

In [3]:
f=lambda x:1/(1+25*x*x)
grid=np.linspace(-1,1,1001)
uniform=np.linspace(-1,1,17);cheb=nos_chebyshev(17)
pu=baricentrica(grid,uniform,f(uniform));pc=baricentrica(grid,cheb,f(cheb))
errors=[np.max(abs(pu-f(grid))),np.max(abs(pc-f(grid)))]
print("erros uniformes/Chebyshev:",errors)
assert errors[1]<errors[0]
plt.plot(grid,f(grid),label="f")
plt.plot(grid,pu,label="uniforme")
plt.plot(grid,pc,label="Chebyshev")
plt.ylim(-0.5,2);plt.xlabel("x");plt.ylabel("valor");plt.legend()
mostrar("05_runge")

erros uniformes/Chebyshev: [np.float64(14.386268436818346), np.float64(0.03670855245148236)]


## 3. Splines e critérios de comparação
Splines cúbicas são polinômios por trechos. Compare o mesmo conjunto uniforme de nós com a interpolação polinomial; defina a condição de contorno usada. Aqui usamos spline natural, segunda derivada nula nas extremidades.

In [4]:
from scipy.interpolate import CubicSpline,BarycentricInterpolator
spline=CubicSpline(uniform,f(uniform),bc_type="natural")
reference=BarycentricInterpolator(cheb,f(cheb))
assert np.allclose(reference(grid),pc,atol=1e-11)
plt.semilogy(grid,np.maximum(abs(pu-f(grid)),1e-16),label="polinômio uniforme")
plt.semilogy(grid,np.maximum(abs(pc-f(grid)),1e-16),label="polinômio Chebyshev")
plt.semilogy(grid,np.maximum(abs(spline(grid)-f(grid)),1e-16),label="spline natural uniforme")
plt.xlabel("x");plt.ylabel("erro absoluto");plt.legend()
mostrar("05_erros")
print("erro máximo spline:",np.max(abs(spline(grid)-f(grid))))

erro máximo spline: 0.003744048308585768


## 4. Fórmula simbólica opcional e compatibilidade
A substituição de `np.math.factorial` por `math.factorial` evita depender de um atributo removido. No acervo, a falha estava registrada na saída da célula de derivadas.

In [5]:
import math
import sympy as sp
x=sp.symbols("x")
polynomial=sp.interpolate(list(zip(nodes.astype(int),values.astype(int))),x)
print("polinômio simbólico:",sp.expand(polynomial))
print("4! =",math.factorial(4))
assert sp.expand(polynomial-(x**3-2*x+1))==0

polinômio simbólico: x**3 - 2*x + 1
4! = 24


## Para estudar
Explique por que a fórmula baricêntrica precisa retornar diretamente yᵢ quando x=xᵢ. Compare graus, tipos de nós e extrapolação. Maior grau não garante menor erro em uma malha uniforme.

## Figuras desta execução

![05_erros](../assets/figuras/05_erros.svg)

![05_runge](../assets/figuras/05_runge.svg)